# LinkedIn Job Description Generation

**Goal.** Take structured job inputs (title, location, skills, experience level,
industry) and produce a natural-language job description. We compare three
approaches side by side:

1. **Zero-shot FLAN-T5-small** — the pretrained model used without any task-specific training.
2. **Fine-tuned FLAN-T5-small** — the same model fine-tuned on a cleaned subset of LinkedIn postings.
3. **Character-level Transformer** — a small Transformer decoder built entirely from scratch,
   using the same architecture as the Tiny Shakespeare exercise.

**Why FLAN-T5-small?** It is a sequence-to-sequence (encoder-decoder) Transformer pretrained
by Google on a large mixture of instruction-following tasks. "Small" means it fits in 6 GB
of VRAM with mixed precision.

**Sections:**
1. Setup
2. Load and clean data
3. Build prompts and targets
4. Tokenize and build a PyTorch Dataset
5. Fine-tune FLAN-T5-small
6. Character-level Transformer from scratch
7. Compare all models

## Section 1 — Setup

Libraries used in this notebook:

- **`torch`** + **`torch.nn`** — PyTorch tensors, autograd, model definition, training loop.
- **`transformers`** — HuggingFace library. Provides `T5ForConditionalGeneration` and
  `AutoTokenizer`. Calling `from_pretrained("google/flan-t5-small")` downloads the weights
  and builds the `nn.Module` automatically.
- **`pandas`** / **`numpy`** — CSV loading and data manipulation.
- **`langdetect`** — language identification used in Section 2 to filter non-English postings.
- **`tqdm`** — progress bars.

All dependencies are listed in `requirements.txt`. Install once with
`pip install -r requirements.txt` before running the notebook.

In [1]:
# Standard library
import os
import re
import random
from pathlib import Path
import math
import json
import time

# Numerical / data
import numpy as np
import pandas as pd

# PyTorch
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

# HuggingFace transformers
from transformers import (
    AutoTokenizer,
    T5ForConditionalGeneration,
    get_linear_schedule_with_warmup,
)

# Language detection (used in Section 2 to filter English-only postings)
from langdetect import detect, DetectorFactory
DetectorFactory.seed = 0  # makes langdetect deterministic across runs

# Progress bars
from tqdm.auto import tqdm
tqdm.pandas()  # enables df.progress_apply(...) for the slow language-detection step

print("Imports OK")

Imports OK


In [2]:
# Reproducibility: fix every RNG we touch. With seed=42 the train/val/test split,
# the 200k subsample, and the optimizer's stochastic behavior will all be repeatable.
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# Paths. Everything lives next to the notebook.
DATA_DIR = Path("./data")                              # the 11 CSVs you already have
CHECKPOINT_DIR = Path("./checkpoint")                  # fine-tuned FLAN-T5-small (Section 6)
CHECKPOINT_SCRATCH_DIR = Path("./checkpoint_scratch")  # from-scratch comparison model (Section 7)
CHECKPOINT_DIR.mkdir(exist_ok=True)
CHECKPOINT_SCRATCH_DIR.mkdir(exist_ok=True)

# Device. The GTX 1660 Super has no Tensor Cores, but FP16 still saves memory,
# which is the binding constraint at 6 GB VRAM. We will enable AMP in Section 6.
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")
if DEVICE.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

# Model name. We refer to it by its HuggingFace Hub ID; downloading happens later
# in Section 4 (tokenizer) and Section 5 (weights), the first time it is requested.
MODEL_NAME = "google/flan-t5-small"
print(f"Model: {MODEL_NAME}")

# Sanity check: the data directory should contain the CSVs we will load next.
csvs = sorted(p.name for p in DATA_DIR.glob("*.csv"))
print(f"\nFound {len(csvs)} CSVs in {DATA_DIR}/:")
for name in csvs:
    print(f"  {name}")

GEN_KWARGS = dict(
    max_length=384,
    num_beams=4,
    no_repeat_ngram_size=3,
    early_stopping=True,
)

Device: cuda
GPU: NVIDIA GeForce GTX 1660 SUPER
VRAM: 6.44 GB
Model: google/flan-t5-small

Found 11 CSVs in data/:
  benefits.csv
  companies.csv
  company_industries.csv
  company_specialities.csv
  employee_counts.csv
  industries.csv
  job_industries.csv
  job_skills.csv
  postings.csv
  salaries.csv
  skills.csv


Running the two cells above should print:

- `Imports OK`
- `Device: cuda` with your GPU name and ~6 GB VRAM. If it says `cpu`, your
  CUDA-enabled PyTorch is not installed correctly — training will be very slow.
- A list of the CSV files found in `./data/`.

## Section 2 — Load and clean data

Loads the raw CSVs and builds a clean DataFrame. Steps:

1. Load only the columns we need from `postings.csv`.
2. Drop rows missing `title` or `description`.
3. Keep only postings whose description is between **50 and 700 words**.
4. Keep only **English** postings using `langdetect` on the first 500 characters
   of each description. This step is slow (a few minutes) and runs only once —
   the result is saved to `data/df_cleaned.csv`. On the next kernel restart the
   first cell loads the cache and you can skip the rest of this section.
5. Join human-readable **skill names** and **industry names** from the bridge
   tables onto each posting.

After this section `df` contains one row per posting with the original fields
plus `skills_joined` and `industries_joined`.

In [3]:
CACHE_PATH = DATA_DIR / "df_cleaned.csv"

if CACHE_PATH.exists():
    df = pd.read_csv(CACHE_PATH)
    print(f"Loaded from cache: {len(df):,} rows — skip remaining Section 2 cells and go to Section 3.")
else:
    df = None
    print("No cache found — run the remaining Section 2 cells to build and cache the data.")


No cache found — run the remaining Section 2 cells to build and cache the data.


In [4]:
# Columns we need from postings.csv. Loading only these keeps memory in check --
# postings.csv has ~30 columns and >100k rows.
POSTINGS_COLS = [
    "job_id",
    "title",
    "description",
    "location",
    "formatted_work_type",
    "formatted_experience_level",
]

postings = pd.read_csv(DATA_DIR / "postings.csv", usecols=POSTINGS_COLS)
print(f"Loaded postings.csv: {len(postings):,} rows, {postings.shape[1]} columns")
print("\nMissing-value % per column:")
print((postings.isna().mean() * 100).round(1).sort_values(ascending=False))
postings.head(2)

Loaded postings.csv: 123,849 rows, 6 columns

Missing-value % per column:
formatted_experience_level    23.7
job_id                         0.0
title                          0.0
description                    0.0
location                       0.0
formatted_work_type            0.0
dtype: float64


,job_id,title,description,location,formatted_work_type,formatted_experience_level
0,921716,Marketing Coordinator,Job descriptionA leading real estate firm in N...,"Princeton, NJ",Full-time,NaN
1,1829192,Mental Health Therapist/Counselor,"At Aspen Therapy and Wellness , we are committ...","Fort Collins, CO",Full-time,NaN


In [5]:
# Step 1: drop rows missing title or description.
before = len(postings)
postings = postings.dropna(subset=["title", "description"]).reset_index(drop=True)
print(f"After dropping missing title/description: {len(postings):,} rows  ({before - len(postings):,} dropped)")

# Step 2: keep only descriptions between 50 and 700 words.
# Plain whitespace split is good enough for a length filter -- we're not measuring
# tokens here, just deciding what's reasonable for a job description.
word_counts = postings["description"].str.split().str.len()
before = len(postings)
postings = postings[(word_counts >= 50) & (word_counts <= 700)].reset_index(drop=True)
print(f"After 50-700 word filter:                {len(postings):,} rows  ({before - len(postings):,} dropped)")

After dropping missing title/description: 123,842 rows  (7 dropped)
After 50-700 word filter:                92,130 rows  (31,712 dropped)


In [6]:
# Step 3: keep only English postings.
# langdetect raises LangDetectException on empty/garbled text -- wrap in try/except.
# We classify on the first 500 chars to keep it fast; full descriptions don't
# change the answer in practice.
def is_english(text: str) -> bool:
    try:
        return detect(str(text)[:500]) == "en"
    except Exception:
        return False

print("Detecting language on each description (this is the slow step, ~5 minutes)...")
mask_en = postings["description"].progress_apply(is_english)
before = len(postings)
postings = postings[mask_en].reset_index(drop=True)
print(f"After English-only filter: {len(postings):,} rows  ({before - len(postings):,} dropped)")

Detecting language on each description (this is the slow step, ~5 minutes)...


  0%|          | 0/92130 [00:00<?, ?it/s]

After English-only filter: 91,971 rows  (159 dropped)


### Joining skill names and industry names

`postings.csv` only has `job_id`. The human-readable skill and industry **names** live in separate tables that we reach through bridge tables:

- `job_skills.csv (job_id, skill_abr)` × `skills.csv (skill_abr, skill_name)` → list of skill names per job.
- `job_industries.csv (job_id, industry_id)` × `industries.csv (industry_id, industry_name)` → list of industry names per job.

We aggregate each list into a single comma-separated string per `job_id` so it slots cleanly into our prompt template later. Sorted + deduplicated so the strings are stable across runs.

In [7]:
# Skills: bridge job_id <-> skill_abr <-> skill_name.
job_skills = pd.read_csv(DATA_DIR / "job_skills.csv")
skills     = pd.read_csv(DATA_DIR / "skills.csv")

skills_joined = (
    job_skills.merge(skills, on="skill_abr", how="left")
              .dropna(subset=["skill_name"])
              .groupby("job_id")["skill_name"]
              .apply(lambda s: ", ".join(sorted(set(s))))
              .rename("skills_joined")
              .reset_index()
)
print(f"Skill rows: {len(job_skills):,}  ->  unique jobs with skills: {len(skills_joined):,}")

# Industries: bridge job_id <-> industry_id <-> industry_name.
job_industries = pd.read_csv(DATA_DIR / "job_industries.csv")
industries     = pd.read_csv(DATA_DIR / "industries.csv")

industries_joined = (
    job_industries.merge(industries, on="industry_id", how="left")
                  .dropna(subset=["industry_name"])
                  .groupby("job_id")["industry_name"]
                  .apply(lambda s: ", ".join(sorted(set(s))))
                  .rename("industries_joined")
                  .reset_index()
)
print(f"Industry rows: {len(job_industries):,}  ->  unique jobs with industries: {len(industries_joined):,}")

# Left-join onto our filtered postings so a job with no skills/industries is kept
# (those columns become NaN and later get rewritten as "not specified").
df = (
    postings
    .merge(skills_joined,     on="job_id", how="left")
    .merge(industries_joined, on="job_id", how="left")
)
print(f"\nFinal cleaned dataframe: {len(df):,} rows, {df.shape[1]} columns")
print(df.columns.tolist())
df.tail()

# Save cache so Section 2 can be skipped on future kernel restarts.
df.to_csv(CACHE_PATH, index=False)
print(f"Saved cache -> {CACHE_PATH}  ({CACHE_PATH.stat().st_size / 1e6:.1f} MB)")


Skill rows: 213,768  ->  unique jobs with skills: 126,807
Industry rows: 164,808  ->  unique jobs with industries: 127,025

Final cleaned dataframe: 91,971 rows, 8 columns
['job_id', 'title', 'description', 'location', 'formatted_work_type', 'formatted_experience_level', 'skills_joined', 'industries_joined']
Saved cache -> data\df_cleaned.csv  (275.4 MB)


After running Section 2 the row count shrinks at each filter step. The English
filter typically leaves **80-100k** rows. The final `df` has the posting fields
plus `skills_joined` and `industries_joined`. Missing values in
`formatted_experience_level`, `skills_joined`, or `industries_joined` are
expected and handled in Section 3.

## Section 3 — Build prompts and targets

Converts each row of `df` into a `(prompt, target)` pair for seq2seq training.

**Prompt format.** `build_prompt()` lists the structured fields in a fixed order.
The same function is used verbatim in `app.py` — any divergence between training
and inference prompts silently hurts quality.

**Missing values.** NaN fields are converted to `"not specified"` inside `build_prompt()`.

**Target.** The raw description with whitespace collapsed.

**Subsample and split.** Up to 200k rows randomly sampled, then split 80/10/10
into `train_df`, `val_df`, `test_df` with `seed=42`.

In [8]:
# --- prompt-building utilities ----------------------------------------------
# IMPORTANT: this entire block (helpers + build_prompt) is duplicated VERBATIM
# in app.py. The web UI must produce the exact same prompt strings the model
# saw during training. Any drift here silently degrades inference quality.

def _clean_text(s) -> str:
    """Collapse all whitespace runs into a single space. Keeps a multi-line
    field (e.g. a title with a stray newline) on one line, and is also used
    for the target description."""
    return re.sub(r"\s+", " ", str(s)).strip()

def _or_unspecified(value) -> str:
    """NaN / None / empty string -> 'not specified'. Otherwise the cleaned value."""
    if value is None:
        return "not specified"
    if isinstance(value, float) and pd.isna(value):
        return "not specified"
    s = _clean_text(value)
    return s if s else "not specified"

def build_prompt(row) -> str:
    """Build the model input string from a row of df.

    The exact format below is what the model is trained on; app.py must call
    this identical function so inference-time prompts match training prompts.
    """
    return (
        "generate a job description for the following role.\n"
        f"title: {_or_unspecified(row.get('title'))}\n"
        f"work_type: {_or_unspecified(row.get('formatted_work_type'))}\n"
        f"experience: {_or_unspecified(row.get('formatted_experience_level'))}\n"
        f"location: {_or_unspecified(row.get('location'))}\n"
        f"industry: {_or_unspecified(row.get('industries_joined'))}\n"
        f"skills: {_or_unspecified(row.get('skills_joined'))}"
    )

# Sanity check on the first row.
print(build_prompt(df.iloc[0]))

generate a job description for the following role.
title: Marketing Coordinator
work_type: Full-time
experience: not specified
location: Princeton, NJ
industry: Real Estate
skills: Marketing, Sales


In [9]:
# Subsample. The spec says 200k; if we have fewer rows we just use everything.
N_TARGET = 200_000
n = min(N_TARGET, len(df))
sampled = df.sample(n=n, random_state=SEED).reset_index(drop=True)
print(f"Sampled {len(sampled):,} rows from {len(df):,}")

# Build (prompt, target) pairs for the sampled rows.
print("Building (prompt, target) pairs...")
sampled["prompt"] = sampled.apply(build_prompt, axis=1)
sampled["target"] = sampled["description"].apply(_clean_text)

# 80/10/10 split. Shuffle once with a fixed seed, then slice.
shuffled = sampled.sample(frac=1.0, random_state=SEED).reset_index(drop=True)
n_train = int(0.80 * len(shuffled))
n_val   = int(0.10 * len(shuffled))
train_df = shuffled.iloc[:n_train].reset_index(drop=True)
val_df   = shuffled.iloc[n_train:n_train + n_val].reset_index(drop=True)
test_df  = shuffled.iloc[n_train + n_val:].reset_index(drop=True)
print(f"train={len(train_df):,}  val={len(val_df):,}  test={len(test_df):,}")

# Show 3 examples so we can sanity-check the format visually.
for i in range(3):
    print("=" * 80)
    print("PROMPT:")
    print(train_df.iloc[i]["prompt"])
    tgt = train_df.iloc[i]["target"]
    print("\nTARGET (first 300 chars):")
    print(tgt[:300] + ("..." if len(tgt) > 300 else ""))

Sampled 91,971 rows from 91,971
Building (prompt, target) pairs...
train=73,576  val=9,197  test=9,198
PROMPT:
generate a job description for the following role.
title: Medical Technologist/Medical Lab Technician
work_type: Part-time
experience: Mid-Senior level
location: Auburn, WA
industry: Biotechnology Research, Hospitals and Health Care, Research Services
skills: Analyst, Information Technology, Research

TARGET (first 300 chars):
You Belong Here. At MultiCare, we strive to offer a true sense of belonging for all our employees. Across our health care network, you will find a dynamic range of meaningful careers, opportunities for growth, safe workplaces, and flexible schedules. We are connected by our mission - partnering and ...
PROMPT:
generate a job description for the following role.
title: Manufacturing Engineer
work_type: Full-time
experience: Mid-Senior level
location: Germantown, WI
industry: Medical Equipment Manufacturing
skills: Engineering, Information Technology

TARGE

## Section 4 — Tokenize and build a PyTorch Dataset

The model works with integer token IDs, not raw strings. The tokenizer (SentencePiece
for T5) splits text into subword pieces and maps each to an integer.

This section has two cells:

- **First cell** — loads the tokenizer and defines the `JobPostingDataset` class.
  Run this every session — it takes about a second.
- **Second cell** — pre-tokenizes all train/val/test examples. Only needed before
  running Section 5 training. Skip it if you are going straight to Section 6 or 7.

**The `-100` label trick.** `CrossEntropyLoss` ignores positions where the label
is `-100`. We replace padding token IDs in the target with `-100` so the model is
not rewarded for predicting padding.

**Lengths.** `max_input_length=256` covers the prompt. `max_target_length=384`
covers most descriptions.

In [10]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
print(f"Tokenizer: {type(tokenizer).__name__}, vocab size: {tokenizer.vocab_size}")

MAX_INPUT_LEN  = 256
MAX_TARGET_LEN = 384

class JobPostingDataset(Dataset):
    def __init__(self, df, tokenizer, max_input_len=MAX_INPUT_LEN, max_target_len=MAX_TARGET_LEN):
        prompts = df["prompt"].tolist()
        targets = df["target"].tolist()

        print(f"  Pre-tokenizing {len(prompts):,} examples...", end=" ", flush=True)
        enc_in = tokenizer(
            prompts,
            max_length=max_input_len,
            padding="max_length",
            truncation=True,
            return_tensors="pt",
        )
        enc_out = tokenizer(
            targets,
            max_length=max_target_len,
            padding="max_length",
            truncation=True,
            return_tensors="pt",
        )
        self.input_ids      = enc_in["input_ids"]
        self.attention_mask = enc_in["attention_mask"]
        labels = enc_out["input_ids"].clone()
        labels[labels == tokenizer.pad_token_id] = -100
        self.labels = labels
        print("done.")

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, idx):
        return {
            "input_ids":      self.input_ids[idx],
            "attention_mask": self.attention_mask[idx],
            "labels":         self.labels[idx],
        }

Tokenizer: T5Tokenizer, vocab size: 32100


In [21]:
# Only needed if you are about to train Section 6.
# Skip this cell if you are going straight to Section 7 or 8.
print("Building datasets (pre-tokenizing all splits)...")
train_ds = JobPostingDataset(train_df, tokenizer)
val_ds   = JobPostingDataset(val_df,   tokenizer)
test_ds  = JobPostingDataset(test_df,  tokenizer)
print(f"Datasets ready: train={len(train_ds):,}  val={len(val_ds):,}  test={len(test_ds):,}")

sample = train_ds[0]
print("Sample keys:", list(sample.keys()))
print("input_ids shape:", sample["input_ids"].shape, "  labels shape:", sample["labels"].shape)

Building datasets (pre-tokenizing all splits)...
  Pre-tokenizing 73,576 examples... done.
  Pre-tokenizing 9,197 examples... done.
  Pre-tokenizing 9,198 examples... done.
Datasets ready: train=73,576  val=9,197  test=9,198
Sample keys: ['input_ids', 'attention_mask', 'labels']
input_ids shape: torch.Size([256])   labels shape: torch.Size([384])


In [22]:
BATCH_SIZE = 8

# num_workers=0 on Windows avoids spawn issues inside notebooks. pin_memory speeds
# up the host->GPU copy when training on CUDA (no effect on CPU).
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=0, pin_memory=(DEVICE.type == "cuda"))
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=0, pin_memory=(DEVICE.type == "cuda"))
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=0, pin_memory=(DEVICE.type == "cuda"))

# Sanity: pull one batch and confirm shapes.
batch = next(iter(train_loader))
print("Batch shapes:")
for k, v in batch.items():
    print(f"  {k}: {tuple(v.shape)}")

Batch shapes:
  input_ids: (8, 256)
  attention_mask: (8, 256)
  labels: (8, 384)


## Section 5 — Fine-tune FLAN-T5-small

Starts from Google's pretrained FLAN-T5-small weights and continues training
on our job-description data.

**Training setup:**
- AdamW optimizer, `lr=1e-4`, `weight_decay=0.01`
- Linear warmup over the first 500 steps, then linear decay to 0
- 3 epochs, gradient clipping at 1.0, gradient accumulation of 2
  (effective batch size = 16)

**Loss.** Calling `model(input_ids=..., labels=...)` returns `outputs.loss`,
the seq2seq cross-entropy over non-masked positions. We call `.backward()` on it directly.

**Checkpoint.** The best-validation-loss model is saved to `./checkpoint/` at the
end of each epoch. `app.py` loads from this directory at startup.

**Time.** Roughly 6-10 hours on a GTX 1660 Super.

In [23]:
# Hyperparameters.
GRAD_ACCUM_STEPS = 2
N_EPOCHS         = 2
_continuing      = (CHECKPOINT_DIR / "config.json").exists()
LR               = 3e-5 if _continuing else 1e-4
WEIGHT_DECAY     = 0.01
WARMUP_STEPS     = 500
GRAD_CLIP        = 1.0

steps_per_epoch = math.ceil(len(train_loader) / GRAD_ACCUM_STEPS)
total_steps     = steps_per_epoch * N_EPOCHS
print(f"steps/epoch={steps_per_epoch}  total_steps={total_steps}")

# FP16/AMP causes NaN on T5 with this GPU — train in FP32 for stability.
USE_AMP = False

def evaluate(model, loader):
    model.eval()
    total, count = 0.0, 0
    with torch.no_grad():
        for batch in tqdm(loader, desc="val", leave=False):
            batch = {k: v.to(DEVICE, non_blocking=True) for k, v in batch.items()}
            out = model(**batch)
            total += out.loss.item() * batch["input_ids"].size(0)
            count += batch["input_ids"].size(0)
    return total / max(count, 1)

def train_model(checkpoint_dir, model, tag):
    checkpoint_dir = Path(checkpoint_dir)
    checkpoint_dir.mkdir(exist_ok=True)
    best_path = checkpoint_dir / "best_val.json"
    best_val = float("inf")
    if best_path.exists():
        best_val = json.loads(best_path.read_text())["best_val"]
        print(f"[{tag}] resuming with best_val={best_val:.4f}")

    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    scheduler = get_linear_schedule_with_warmup(
        optimizer, num_warmup_steps=WARMUP_STEPS, num_training_steps=total_steps
    )

    global_step = 0
    for epoch in range(N_EPOCHS):
        model.train()
        running_loss = 0.0
        t0 = time.time()
        pbar = tqdm(train_loader, desc=f"[{tag}] epoch {epoch+1}/{N_EPOCHS}")
        optimizer.zero_grad(set_to_none=True)

        for step, batch in enumerate(pbar):
            batch = {k: v.to(DEVICE, non_blocking=True) for k, v in batch.items()}
            out  = model(**batch)
            loss = out.loss / GRAD_ACCUM_STEPS
            loss.backward()
            running_loss += loss.item() * GRAD_ACCUM_STEPS

            if (step + 1) % GRAD_ACCUM_STEPS == 0:
                torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
                optimizer.step()
                scheduler.step()
                optimizer.zero_grad(set_to_none=True)
                global_step += 1

                if global_step % 100 == 0:
                    pbar.set_postfix(
                        loss=f"{running_loss/(step+1):.4f}",
                        lr=f"{scheduler.get_last_lr()[0]:.2e}",
                    )

        avg_train = running_loss / len(train_loader)
        val_loss  = evaluate(model, val_loader)
        dt = time.time() - t0
        print(f"[{tag}] epoch {epoch+1}: train_loss={avg_train:.4f}  val_loss={val_loss:.4f}  ({dt/60:.1f} min)")

        if val_loss < best_val:
            best_val = val_loss
            model.save_pretrained(checkpoint_dir)
            tokenizer.save_pretrained(checkpoint_dir)
            best_path.write_text(json.dumps({"best_val": best_val, "epoch": epoch + 1}))
            print(f"[{tag}]   saved (new best val_loss={best_val:.4f}) -> {checkpoint_dir}")

    return best_val

if _continuing:
    print(f"Continuing from fine-tuned checkpoint in {CHECKPOINT_DIR} (lr={LR:.0e}) ...")
    model_pre = T5ForConditionalGeneration.from_pretrained(CHECKPOINT_DIR).to(DEVICE)
else:
    print(f"Fine-tuning from scratch: {MODEL_NAME} (lr={LR:.0e}) ...")
    model_pre = T5ForConditionalGeneration.from_pretrained(MODEL_NAME).to(DEVICE)
model_pre.config.use_cache = False
print(f"Trainable parameters: {sum(p.numel() for p in model_pre.parameters() if p.requires_grad):,}")

best_pre = train_model(CHECKPOINT_DIR, model_pre, tag="pretrained")
print(f"Pretrained fine-tune done. Best val loss: {best_pre:.4f}")

del model_pre
if DEVICE.type == "cuda":
    torch.cuda.empty_cache()

steps/epoch=4599  total_steps=9198
Continuing from fine-tuned checkpoint in checkpoint (lr=3e-05) ...


Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


Trainable parameters: 76,961,152
[pretrained] resuming with best_val=2.8387


[pretrained] epoch 1/2:   0%|          | 0/9197 [00:00<?, ?it/s]

...

[pretrained] epoch 1: train_loss=3.0699  val_loss=2.8047  (364.0 min)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[pretrained]   saved (new best val_loss=2.8047) -> checkpoint


[pretrained] epoch 2/2:   0%|          | 0/9197 [00:00<?, ?it/s]

KeyboardInterrupt: 

## Section 6 — Character-level Transformer from scratch

Unlike Section 5 which started from pretrained weights, here we train a model
entirely from scratch using only our data.

We concatenate every `prompt + "\n" + description` pair into one long character
stream, then train a causal Transformer decoder to predict the next character at
every position — the same approach used in the Tiny Shakespeare exercise. At
inference time the model is seeded with a prompt and generates the description
one character at a time.

The architecture (`TransformerEmbedding` + `TransformerDecoder`) is taken directly
from the Tiny Shakespeare notebook.

In [1]:
# Subsample — 2 000 examples gives ~1 M characters,
# comparable to the Tiny Shakespeare corpus (1.1 M chars).
N_TRAIN_S7 = 1_500
train_s7 = train_df.iloc[:N_TRAIN_S7].reset_index(drop=True)
print(f"Section 6 subsample: {len(train_s7):,} training examples")

# Build character vocabulary — same as Tiny Shakespeare:
# sorted unique chars -> stoi / itos.
charset = set()
for _, row in train_s7.iterrows():
    charset.update(row["prompt"] + chr(10) + row["description"])

vocabulary = sorted(charset)
stoi = {c: i for i, c in enumerate(vocabulary)}
itos = {i: c for i, c in enumerate(vocabulary)}
vocab_size = len(vocabulary)

print(f"Vocabulary size: {vocab_size} unique characters")
print(vocabulary)

NameError: name 'train_df' is not defined

### 6.1 Training data — sliding window over concatenated text

Identical to Tiny Shakespeare: we join all examples into one long string,
then extract every possible window of `CONTEXT_LEN=128` characters.
Each window `x[i : i+128]` is paired with `x[i+1 : i+129]`.

No stride — every position in the text becomes a training example,
exactly as in the Shakespeare notebook.

In [ ]:
CONTEXT_LEN = 256
STRIDE      = 64   # step between windows; keeps tensor size manageable

# Join all training examples into one long character stream
full_text = chr(10).join(
    row["prompt"] + chr(10) + row["description"]
    for _, row in train_s7.iterrows()
)
print(f"Total characters in training corpus: {len(full_text):,}")

# Encode character by character
encoded = [stoi.get(c, 0) for c in full_text]

# Sliding windows
xs = []
ys = []
for i in range(0, len(encoded) - CONTEXT_LEN, STRIDE):
    xs.append(encoded[i : i + CONTEXT_LEN])
    ys.append(encoded[i + 1 : i + CONTEXT_LEN + 1])

xs_tensor = torch.tensor(xs, dtype=torch.long)
ys_tensor = torch.tensor(ys, dtype=torch.long)
print(f"Shape input:  {xs_tensor.shape}")
print(f"Shape target: {ys_tensor.shape}")

# Shuffle and split 80 / 20
rnd_idxs  = torch.randperm(xs_tensor.shape[0])
xs_tensor = xs_tensor[rnd_idxs]
ys_tensor = ys_tensor[rnd_idxs]
split = int(xs_tensor.shape[0] * 0.8)
x_train_s7, y_train_s7 = xs_tensor[:split], ys_tensor[:split]
x_test_s7,  y_test_s7  = xs_tensor[split:],  ys_tensor[split:]
print(f"Train: {x_train_s7.shape[0]:,}   Test: {x_test_s7.shape[0]:,}")

### 6.2 Model — causal Transformer decoder

Same architecture and hyperparameters as the Tiny Shakespeare notebook:

- **`TransformerEmbedding`** — token embedding + learned positional embedding
- **`TransformerDecoder`** — 4 layers of `nn.TransformerEncoderLayer` with a
  causal mask, ending with a linear classifier over the character vocabulary

Generation uses **top-p (nucleus) sampling**: at each step only the most
probable tokens whose cumulative probability exceeds `p` are kept, then one
is sampled. This produces more varied output than greedy or beam search.

In [ ]:
class TransformerEmbedding(nn.Module):
    def __init__(self, embedding_dim, vocab_size, sequence_len):
        super().__init__()
        self.token_embedding    = nn.Embedding(vocab_size, embedding_dim)
        self.position_embedding = nn.Embedding(sequence_len, embedding_dim)
        self.register_buffer("positions", torch.arange(sequence_len).reshape(1, -1))

    def forward(self, x):
        seq_len = x.shape[1]
        return (self.token_embedding(x)
                + self.position_embedding(self.positions[:, :seq_len]))


class TransformerDecoder(nn.Module):
    def __init__(self, vocab_size, sequence_len, embedding_dim,
                 n_layer, dropout, mlp_dim, nhead):
        super().__init__()
        self.embedding = TransformerEmbedding(embedding_dim, vocab_size, sequence_len)
        block = nn.TransformerEncoderLayer(
            d_model=embedding_dim, nhead=nhead,
            dim_feedforward=mlp_dim, dropout=dropout, batch_first=True,
        )
        self.transformer = nn.TransformerEncoder(block, num_layers=n_layer)
        self.classifier  = nn.Linear(embedding_dim, vocab_size)

    def forward(self, x):
        seq_len = x.shape[1]
        x = self.embedding(x)
        causal_mask = nn.Transformer.generate_square_subsequent_mask(seq_len).to(x.device)
        x = self.transformer(x, mask=causal_mask, is_causal=True)
        return self.classifier(x)

In [ ]:
# Exact same hyperparameters as the Tiny Shakespeare notebook
EMBEDDING_DIM = 256
N_HEADS       = 8
N_LAYER       = 4
DROPOUT       = 0.2
MLP_DIM       = 512

transformer_char = TransformerDecoder(
    vocab_size=vocab_size,
    sequence_len=CONTEXT_LEN,
    embedding_dim=EMBEDDING_DIM,
    n_layer=N_LAYER,
    dropout=DROPOUT,
    mlp_dim=MLP_DIM,
    nhead=N_HEADS,
)


# Top-p sampling — identical to Tiny Shakespeare
def sample_top_p(logits, p=0.9, temperature=1.0):
    temperature = max(temperature, 1e-5)
    scaled_logits = logits / temperature
    sorted_logits, sorted_indices = torch.sort(scaled_logits, descending=True)
    cumulative_probs = torch.cumsum(F.softmax(sorted_logits, dim=-1), dim=-1)
    sorted_indices_to_remove = cumulative_probs > p
    sorted_indices_to_remove[..., 1:] = sorted_indices_to_remove[..., :-1].clone()
    sorted_indices_to_remove[..., 0] = False
    filtered_logits = scaled_logits.clone()
    indices_to_remove = sorted_indices_to_remove.scatter(
        0, sorted_indices, sorted_indices_to_remove
    )
    filtered_logits[indices_to_remove] = float("-inf")
    probs = F.softmax(filtered_logits, dim=-1)
    return torch.multinomial(probs, num_samples=1)


# Generation — identical to Tiny Shakespeare
def generate_text_s7(model, initial_text="", temperature=1.0,
                     p=0.9, length=400, device=DEVICE):
    model.eval()
    encoded_text = [stoi[c] for c in initial_text if c in stoi]
    if not encoded_text:
        encoded_text = [stoi[vocabulary[0]]]
    with torch.no_grad():
        while len(encoded_text) < length:
            context = encoded_text[-CONTEXT_LEN:]
            x = torch.tensor(context, dtype=torch.long, device=device).reshape(1, -1)
            y = model(x)
            logits = y[0, -1, :].to("cpu")
            next_token = sample_top_p(logits, p=p, temperature=temperature)
            encoded_text.append(next_token.item())
    return "".join(itos[i] for i in encoded_text)


_sample_prompt = build_prompt(train_s7.iloc[0])

In [ ]:
# Exact same training setup as Tiny Shakespeare
EPOCHS_S7         = 10
BATCH_SIZE_S7     = 64
BATCHES_PER_EPOCH = 500
LEARNING_RATE_S7  = 3e-4

transformer_char.to(DEVICE)
criterion_s7 = nn.CrossEntropyLoss()
optimizer_s7 = torch.optim.Adam(transformer_char.parameters(), lr=LEARNING_RATE_S7)

pbar = tqdm(range(EPOCHS_S7), desc="epoch")
for epoch in pbar:
    rand_idxs = torch.randperm(x_train_s7.shape[0])
    x_perm    = x_train_s7[rand_idxs]
    y_perm    = y_train_s7[rand_idxs]

    print(generate_text_s7(transformer_char, _sample_prompt, length=200))

    transformer_char.train()
    losses = []
    for batch_idx, idx in enumerate(range(0, len(x_perm), BATCH_SIZE_S7)):
        if batch_idx >= BATCHES_PER_EPOCH:
            break
        x_batch = x_perm[idx : idx + BATCH_SIZE_S7].to(DEVICE)
        y_batch = y_perm[idx : idx + BATCH_SIZE_S7].to(DEVICE)

        optimizer_s7.zero_grad()
        preds = transformer_char(x_batch)
        loss  = criterion_s7(preds.permute(0, 2, 1), y_batch)
        loss.backward()
        optimizer_s7.step()

        losses.append(loss.item())
        pbar.set_description(f"epoch {epoch + 1} loss = {sum(losses)/len(losses):.4f}")

# Save weights and vocabulary
TRANSFORMER_PATH = CHECKPOINT_SCRATCH_DIR / "char_transformer.pt"
torch.save(transformer_char.state_dict(), TRANSFORMER_PATH)

_vocab_data = {
    "itos":          vocabulary,
    "context_len":   CONTEXT_LEN,
    "embedding_dim": EMBEDDING_DIM,
    "n_heads":       N_HEADS,
    "n_layer":       N_LAYER,
    "dropout":       DROPOUT,
    "mlp_dim":       MLP_DIM,
}
(CHECKPOINT_SCRATCH_DIR / "char_vocab.json").write_text(json.dumps(_vocab_data))
print(f"Saved model -> {TRANSFORMER_PATH}")
print(f"Saved vocab -> {CHECKPOINT_SCRATCH_DIR / chr(39)}char_vocab.json{chr(39)}")

## Section 7 — Compare all models

Runs the same prompt through all three trained models and prints the outputs
side by side:

1. **Zero-shot FLAN-T5-small** — pretrained weights only, no fine-tuning.
2. **Fine-tuned FLAN-T5-small** (Section 5) — loaded from `./checkpoint/`.
3. **Character-level Transformer** (Section 6) — loaded from
   `./checkpoint_scratch/char_transformer.pt`.

The char Transformer requires `char_vocab.json` in the same directory
(written at the end of Section 6).

In [ ]:
# --- [1/2] T5 models -------------------------------------------------------
@torch.no_grad()
def generate_t5(model, prompt: str) -> str:
    enc = tokenizer(prompt, max_length=MAX_INPUT_LEN, truncation=True,
                    return_tensors="pt").to(DEVICE)
    return tokenizer.decode(model.generate(**enc, **GEN_KWARGS)[0],
                            skip_special_tokens=True)

print("Loading fine-tuned FLAN-T5 checkpoint...")
model_pre_eval = T5ForConditionalGeneration.from_pretrained(CHECKPOINT_DIR).to(DEVICE).eval()
print("Done.\n")

# --- [3] Char-level Transformer (Section 6) --------------------------------
_na          = "(model not available – run Section 6 first)"
generate_char = None

_transf_path = CHECKPOINT_SCRATCH_DIR / "char_transformer.pt"
_vocab_path  = CHECKPOINT_SCRATCH_DIR / "char_vocab.json"

if _transf_path.exists() and _vocab_path.exists():
    _vd        = json.loads(_vocab_path.read_text())
    _itos_list = _vd["itos"]
    _stoi_s8   = {c: i for i, c in enumerate(_itos_list)}
    _itos_s8   = {i: c for i, c in enumerate(_itos_list)}
    _VOCAB_S8  = len(_itos_list)
    _CTX_S8    = _vd["context_len"]

    _transf_s8 = TransformerDecoder(
        vocab_size=_VOCAB_S8,  sequence_len=_CTX_S8,
        embedding_dim=_vd["embedding_dim"], n_layer=_vd["n_layer"],
        dropout=_vd.get("dropout", 0.2),   mlp_dim=_vd["mlp_dim"],
        nhead=_vd["n_heads"],
    )
    _transf_s8.load_state_dict(torch.load(str(_transf_path), map_location=DEVICE))
    _transf_s8.to(DEVICE).eval()
    print("Char Transformer loaded from disk.\n")

    @torch.no_grad()
    def generate_char(prompt: str, max_new: int = 400,
                      temperature: float = 0.8) -> str:
        seed = [_stoi_s8.get(c, 0) for c in prompt + "\n"]
        for _ in range(max_new):
            context = seed[-_CTX_S8:]
            x = torch.tensor(context, dtype=torch.long, device=DEVICE).unsqueeze(0)
            logits = _transf_s8(x)[0, -1, :]
            probs  = torch.softmax(logits / max(temperature, 1e-6), dim=-1).cpu().numpy()
            next_id = int(np.random.choice(_VOCAB_S8, p=probs))
            seed.append(next_id)
        return "".join(_itos_s8.get(i, "") for i in seed[len(prompt) + 1:])

# --- Comparison -------------------------------------------------------------
sample_row    = val_df.sample(1, random_state=SEED).iloc[0]
sample_prompt = build_prompt(sample_row)

print("Prompt:\n", sample_prompt, "\n")
print("Ground truth (first 400 chars):\n", sample_row["description"][:400], "\n")

base_model_s8 = T5ForConditionalGeneration.from_pretrained(MODEL_NAME).to(DEVICE).eval()

print("=" * 60)
print("[1] Zero-shot FLAN-T5-small:")
print(generate_t5(base_model_s8, sample_prompt))
print()
print("=" * 60)
print("[2] Fine-tuned FLAN-T5-small (Section 5):")
print(generate_t5(model_pre_eval, sample_prompt))
print()
print("=" * 60)
print("[3] Char-level Transformer (Section 6):")
print(generate_char(sample_prompt) if generate_char else _na)

Loading fine-tuned FLAN-T5 checkpoint...


Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


Done.

Char Transformer loaded from disk.

Prompt:
 generate a job description for the following role.
title: Registered Dental Hygienist
work_type: Part-time
experience: not specified
location: Marlborough, MA
industry: not specified
skills: Health Care Provider 

Ground truth (first 400 chars):
 Company Description We are open Monday-Thursday from 8:00am-5:00pm with lunch from 1:00pm-2:00pm. NO nights or weekends. 
We offer competitive compensation. 401k and profit share, paid time off, CEU reimbursment, employee discount for dental work.

 Role Description This is a part-time/full time on-site role as a Registered Dental Hygienist at Todd Reeves DMD located in Marlborough, MA. As a Regis 



Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


[1] Zero-shot FLAN-T5-small:
Registered Dental Hygienist working in the dental industry.

[2] Fine-tuned FLAN-T5-small (Section 5):
Job Title: Registered Dental HygienistLocation: Marlborough, MA (Onsite)Duration: Part-Time (On-Site) Job Description:As a Registered Hygiene Technician, you will be responsible for providing dental care to patients and their families. You will work closely with the dental team to ensure that the patient’s dental needs are met and that they are properly hygienized. Qualifications:Bachelor’s degree in dental hygiene, preferably in a related field.Experience in oral hygiene is a plus.Excellent communication and interpersonal skills.Ability to work independently and as part of a team.Strong organizational and time management skills.

[3] Char-level Transformer (Section 6):
guskil mer work and ld desene of wrkd reanated mecurvint aced theccr wle then fesscal! Surapplonin EOpertante Satons: Weror Hest Recalumerety che ne as, fullolonged tha or lenarat d to prop